# 00 · Setup and data download (Colab)

Downloads the Roboflow dataset into `MyDrive/cell-image-counting/data/`. Run once per Drive; the other notebooks reuse it.

**Before running:** upload this folder to Drive (see `README.md`), add the `ROBOFLOW_EXPORT_URL` secret (key icon, notebook access on), then Run all. No GPU needed here.

In [ ]:
# --- Colab bootstrap (identical in every notebook in this folder, except EXTRAS) ---
# Expects the contents of notebooks/notebooks_colab_draft/ uploaded to DRIVE_ROOT on Google Drive:
# pyproject.toml, src/cellcount/, splits/splits.csv and these notebooks (see README.md).
# data/ and outputs/ are created there too, so they survive runtime resets.
# Secrets (key icon in the left sidebar, notebook access on): ROBOFLOW_EXPORT_URL.
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/cell-image-counting"
EXTRAS = ""  # "[model]" adds torch, ultralytics and cellpose

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    root = Path(DRIVE_ROOT)
    missing = [p for p in ("pyproject.toml", "src/cellcount/__init__.py", "splits/splits.csv") if not (root / p).exists()]
    if missing:
        raise FileNotFoundError(f"missing in {root}: {missing}. Upload notebooks_colab_draft/ there (README.md).")
    # Install from a local copy: Drive is slow and pip would write build files into it.
    # Re-copied on every run, so re-uploaded code is picked up after a runtime restart.
    code_dir = Path("/content/cellcount_code")
    shutil.rmtree(code_dir, ignore_errors=True)
    shutil.copytree(root / "src", code_dir / "src", ignore=shutil.ignore_patterns("__pycache__", "*.egg-info"))
    shutil.copy(root / "pyproject.toml", code_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{code_dir}{EXTRAS}"], check=True)
    sys.path.insert(0, str(code_dir / "src"))  # import without restarting the runtime
    os.environ["CELLCOUNT_STORAGE"] = DRIVE_ROOT  # data/, outputs/ and splits/ live here
    os.chdir("/content")  # pretrained weights (yolo26n.pt, ...) download here, not to Drive

## Environment check

In [ ]:
import platform
from cellcount.config import get_paths, is_colab

paths = get_paths()
print("Python   :", platform.python_version())
print("On Colab :", is_colab())
print("Data     :", paths.data)
print("Outputs  :", paths.outputs)

## Download

The default source is the **raw export link** of our Roboflow fork: the 500 original images, with no resizing or augmentation, stored as the `ROBOFLOW_EXPORT_URL` secret.

Don't use the public project's version 1 (~1,200 images). Its 350 training images are augmented 3× and every image is stretched to 640×640, so it holds only 500 unique images.

Once someone generates a version in the fork (Roboflow > Generate, with no preprocessing or augmentation), the API route below works too: set `API_VERSION` and add the `ROBOFLOW_API_KEY` secret.

In [ ]:
from cellcount.config import get_secret
from cellcount.data import download_export, download_roboflow

API_VERSION = None  # e.g. 1, once a version is generated in the fork

if API_VERSION is None:
    root = download_export(get_secret("ROBOFLOW_EXPORT_URL"), paths.data)
else:
    root = download_roboflow(get_secret("ROBOFLOW_API_KEY"), paths.data, version=API_VERSION)
root

## Sanity check

In [ ]:
from cellcount.data import load_coco

images, boxes = load_coco(root)
print(images.groupby("split").size().to_string(), "\n")
print("classes:", boxes["category"].value_counts().to_dict())
print("cells per image:", images["count"].describe().round(1).to_dict())

Next: `02_baselines.ipynb` (reproduces the baselines on a GPU) or `03_yolo_finetune.ipynb` (trains new detectors).